In [3]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

In [4]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
#Normalization
x_train=x_train.astype('float32')/255.0
x_test=x_test.astype('float32')/255.0

In [5]:
#Set canvas at 75X75
CANVAS=75
#Set CIFAR 10 at 32X32
OBJ=28


In [6]:
def make_localized_image(mnist_img):
    canvas=np.zeros((CANVAS,CANVAS),dtype=np.float32)

    x=np.random.randint(0,CANVAS-OBJ+1)
    y=np.random.randint(0,CANVAS-OBJ+1)
    second_corr_x=x+OBJ
    second_corr_y=y+OBJ
    canvas[x:x+OBJ,y:y+OBJ]=mnist_img
    bbox=np.array([x/CANVAS,y/CANVAS,second_corr_x/CANVAS,second_corr_y/CANVAS]) # Normalized Bounding Box 0 to 1
    canvas=np.expand_dims(canvas,axis=-1)
    return canvas,bbox






In [7]:
n_train=12000
n_test=2000

train_images=[]
test_images=[]
train_boxes=[]
test_boxes=[]
train_labels=[]
test_labels=[]
for i in range(n_train):
    img,box=make_localized_image(x_train[i])
    train_images.append(img)
    train_boxes.append(box)
    train_labels.append(y_train[i])

for i in range(n_test):
    img,box=make_localized_image(x_test[i])
    test_images.append(img)
    test_boxes.append(box)
    test_labels.append(y_test[i])


In [8]:
train_images=np.array(train_images,dtype=np.float32)
test_images=np.array(test_images,dtype=np.float32)
train_boxes=np.array(train_boxes,dtype=np.float32)
test_boxes=np.array(test_boxes,dtype=np.float32)
train_labels=np.array(train_labels,dtype=np.int32)
test_labels=np.array(test_labels,dtype=np.int32)

In [9]:
inputs=tf.keras.Input(shape=(CANVAS,CANVAS,1))

x=tf.keras.layers.Conv2D(16,3,activation='relu')(inputs)
x=tf.keras.layers.MaxPooling2D()(x)

x=tf.keras.layers.Conv2D(32,3,activation='relu')(x)
x=tf.keras.layers.MaxPooling2D()(x)

x=tf.keras.layers.Conv2D(64,3,activation='relu')(x)
x=tf.keras.layers.Flatten()(x)

x=tf.keras.layers.Dense(128,activation='relu')(x)

class_output=tf.keras.layers.Dense(10,activation='softmax',name='class_output')(x)
bbox_output=tf.keras.layers.Dense(4,activation='sigmoid',name='bbox_output')(x)

model=tf.keras.Model(inputs=inputs,outputs=[class_output,bbox_output])



I0000 00:00:1772021110.439054  105067 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 2766 MB memory:  -> device: 0, name: NVIDIA GeForce GTX 1650, pci bus id: 0000:01:00.0, compute capability: 7.5


In [10]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 75, 75, 1) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 73, 73,    │        160 │ input_layer[0][0] │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 36, 36,    │          0 │ conv2d[0][0]      │
│ (MaxPooling2D)      │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 34, 34,    │      4,640 │ max_pooling2d[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 17, 17,    │          0 │ conv2d_1[0][0]    │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 15, 15,    │     18,496 │ max_pooling2d_1[… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 14400)     │          0 │ conv2d_2[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 128)       │  1,843,328 │ flatten[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ class_output        │ (None, 10)        │      1,290 │ dense[0][0]       │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bbox_output (Dense) │ (None, 4)         │        516 │ dense[0][0]       │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,868,430 (7.13 MB)

 Trainable params: 1,868,430 (7.13 MB)

 Non-trainable params: 0 (0.00 B)

In [11]:
train_ds=tf.data.Dataset.from_tensor_slices((train_images,{'class_output':train_labels,'bbox_output':train_boxes}))
train_ds=train_ds.shuffle(1000)
train_ds=train_ds.batch(32)
test_ds=tf.data.Dataset.from_tensor_slices((test_images,{'class_output':test_labels,'bbox_output':test_boxes}))
test_ds=test_ds.shuffle(1000)
test_ds=test_ds.batch(32)

2026-02-25 14:05:12.418522: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 270000000 exceeds 10% of free system memory.
2026-02-25 14:05:12.687379: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 270000000 exceeds 10% of free system memory.
2026-02-25 14:05:12.818310: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 45000000 exceeds 10% of free system memory.
2026-02-25 14:05:12.830210: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 45000000 exceeds 10% of free system memory.


In [12]:
def iou_metric(y_true,y_pred):
    x1_t=y_true[:,0]
    y1_t=y_true[:,1]
    x2_t=y_true[:,2]
    y2_t=y_true[:,3]

    x1_p=y_pred[:,0]
    y1_p=y_pred[:,1]
    x2_p=y_pred[:,2]
    y2_p=y_pred[:,3]

    x1_inter=tf.maximum(x1_t,x1_p)
    y1_inter=tf.maximum(y1_t,y1_p)
    x2_inter=tf.minimum(x2_t,x2_p)
    y2_inter=tf.minimum(y2_t,y2_p)

    inter_width=tf.maximum(0.0,x2_inter-x1_inter)
    inter_height=tf.maximum(0.0,y2_inter-y1_inter)
    inter_area=inter_width*inter_height

    true_area=(x2_t-x1_t)*(y2_t-y1_t)
    pred_area=(x2_p-x1_p)*(y2_p-y1_p)
    union_area=true_area+pred_area-inter_area
    iou=inter_area/(union_area+1e-6)
    return tf.reduce_mean(iou)



In [13]:
model.compile(optimizer='adam',
              loss={'class_output':'sparse_categorical_crossentropy','bbox_output':'mse'},
              metrics={'bbox_output':['mae',iou_metric],'class_output':'accuracy'})

In [14]:
model.fit(train_ds,validation_data=test_ds,epochs=20)

Epoch 1/20


2026-02-25 14:05:12.870539: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 270000000 exceeds 10% of free system memory.
2026-02-25 14:05:14.369991: I external/local_xla/xla/service/service.cc:163] XLA service 0x7a74e800e650 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-02-25 14:05:14.370006: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce GTX 1650, Compute Capability 7.5
2026-02-25 14:05:14.427801: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-02-25 14:05:14.709942: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91801
2026-02-25 14:05:17.869955: I external/local_xla/xla/service/gpu/autotuning/conv_algorithm_picker.cc:546] Omitted potentially buggy algorithm eng14{k25=2} for conv (f32[32,16,73,73]{3,2,1,0}, u8[0]{0

  9/375 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - bbox_output_iou_metric: 5.8218e-04 - bbox_output_loss: 0.0644 - bbox_output_mae: 0.2093 - class_output_accuracy: 0.0796 - class_output_loss: 2.3109 - loss: 2.3752     

I0000 00:00:1772021122.623967  105243 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


370/375 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - bbox_output_iou_metric: 0.3534 - bbox_output_loss: 0.0223 - bbox_output_mae: 0.1087 - class_output_accuracy: 0.2834 - class_output_loss: 1.8943 - loss: 1.9165

2026-02-25 14:05:25.554649: I external/local_xla/xla/service/gpu/autotuning/conv_algorithm_picker.cc:546] Omitted potentially buggy algorithm eng14{k25=2} for conv (f32[32,16,73,73]{3,2,1,0}, u8[0]{0}) custom-call(f32[32,1,75,75]{3,2,1,0}, f32[16,1,3,3]{3,2,1,0}, f32[16]{0}), window={size=3x3}, dim_labels=bf01_oi01->bf01, custom_call_target="__cudnn$convBiasActivationForward", backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"cudnn_conv_backend_config":{"activation_mode":"kRelu","conv_result_scale":1,"side_input_scale":0,"leakyrelu_alpha":0},"force_earliest_schedule":false,"reification_cost":[]}
2026-02-25 14:05:25.599157: I external/local_xla/xla/service/gpu/autotuning/conv_algorithm_picker.cc:546] Omitted potentially buggy algorithm eng14{k25=2} for conv (f32[32,32,34,34]{3,2,1,0}, u8[0]{0}) custom-call(f32[32,16,36,36]{3,2,1,0}, f32[32,16,3,3]{3,2,1,0}, f32[32]{0}), window={size=3x3}, dim_labels=bf01_oi01->bf01, custom_call_target="__cudnn$convBiasActivationFor

375/375 ━━━━━━━━━━━━━━━━━━━━ 14s 11ms/step - bbox_output_iou_metric: 0.4797 - bbox_output_loss: 0.0112 - bbox_output_mae: 0.0777 - class_output_accuracy: 0.4895 - class_output_loss: 1.3828 - loss: 1.3940 - val_bbox_output_iou_metric: 0.5751 - val_bbox_output_loss: 0.0050 - val_bbox_output_mae: 0.0555 - val_class_output_accuracy: 0.7035 - val_class_output_loss: 0.7936 - val_loss: 0.8004
Epoch 2/20
375/375 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - bbox_output_iou_metric: 0.5989 - bbox_output_loss: 0.0044 - bbox_output_mae: 0.0518 - class_output_accuracy: 0.8789 - class_output_loss: 0.3751 - loss: 0.3794 - val_bbox_output_iou_metric: 0.6142 - val_bbox_output_loss: 0.0036 - val_bbox_output_mae: 0.0473 - val_class_output_accuracy: 0.8525 - val_class_output_loss: 0.4695 - val_loss: 0.4725
Epoch 3/20
375/375 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - bbox_output_iou_metric: 0.6484 - bbox_output_loss: 0.0030 - bbox_output_mae: 0.0434 - class_output_accuracy: 0.9473 - class_output_loss: 0.1638 - loss: 0.1669 

In [15]:
idx=5
img=test_images[idx]
true_label=test_labels[idx]
true_bbox=test_boxes[idx]


In [16]:
class_pred,bbox_pred=model.predict(np.expand_dims(img,axis=0))

2026-02-25 14:06:18.418491: I external/local_xla/xla/service/gpu/autotuning/conv_algorithm_picker.cc:546] Omitted potentially buggy algorithm eng14{k25=2} for conv (f32[1,32,34,34]{3,2,1,0}, u8[0]{0}) custom-call(f32[1,16,36,36]{3,2,1,0}, f32[32,16,3,3]{3,2,1,0}, f32[32]{0}), window={size=3x3}, dim_labels=bf01_oi01->bf01, custom_call_target="__cudnn$convBiasActivationForward", backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"cudnn_conv_backend_config":{"activation_mode":"kRelu","conv_result_scale":1,"side_input_scale":0,"leakyrelu_alpha":0},"force_earliest_schedule":false,"reification_cost":[]}
2026-02-25 14:06:18.502685: I external/local_xla/xla/service/gpu/autotuning/conv_algorithm_picker.cc:546] Omitted potentially buggy algorithm eng14{k25=2} for conv (f32[1,64,15,15]{3,2,1,0}, u8[0]{0}) custom-call(f32[1,32,17,17]{3,2,1,0}, f32[64,32,3,3]{3,2,1,0}, f32[64]{0}), window={size=3x3}, dim_labels=bf01_oi01->bf01, custom_call_target="__cudnn$convBiasActivationForwa

1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step


In [17]:
pred_label=np.argmax(class_pred)
pred_label

np.int64(1)

In [18]:
true_label

np.int32(1)